# Building and Running a Pipeline with TSUT

This notebook walks through the core workflow of the **Time Series Unified Toolbox (TSUT)**:

1. **Discover** available nodes via the registry
2. **Configure** each node (data sources, transforms, models, metrics)
3. **Assemble** nodes and edges into a `Pipeline`
4. **Run** training, inference, and evaluation through a `SmartRunner`

## 1. Setup and Node Discovery

TSUT auto-discovers all built-in component nodes at import time. The global `NODE_REGISTRY` holds every registered node type. We configure logging to see what happens under the hood.

In [1]:
%matplotlib ipympl
import sys

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure

configure(level="DEBUG", stream=sys.stdout, fmt="text")

NodeRegistry with 0 registered nodes: []
Successfully registered nodes from module: tsut.components.nodes.data_sources._register
Successfully registered nodes from module: tsut.components.nodes.metrics.classification._register
Successfully registered nodes from module: tsut.components.nodes.metrics.regression._register
Successfully registered nodes from module: tsut.components.nodes.models._register
Successfully registered nodes from module: tsut.components.nodes.sinks._register
Successfully registered nodes from module: tsut.components.nodes.transforms.encodings._register
Successfully registered nodes from module: tsut.components.nodes.transforms.feature_selection._register
Successfully registered nodes from module: tsut.components.nodes.transforms.filters._register
Successfully registered nodes from module: tsut.components.nodes.transforms.imputations._register
Successfully registered nodes from module: tsut.components.nodes.transforms.operations._register
Successfully registered nod

<Logger tsut (DEBUG)>

In [2]:
NODE_REGISTRY.list()

## 2. Configuring Data Source Nodes

Each node type has a **Config** class (retrievable from the registry) that bundles:
- **`hyperparameters`** -- tuneable parameters for hyperparameter search
- **`running_config`** -- execution-time settings (file paths, backend options, etc.)
- **`in_ports` / `out_ports`** -- the data contract (array type, shape, category)

Here we configure two `TabularCSVFetcher` sources: one for features and one for targets.

In [3]:
csv_config_class = NODE_REGISTRY.get_node_config_class("TabularCSVFetcher")
csv_running_config_class = NODE_REGISTRY["TabularCSVFetcher"]["running_config"]

In [4]:
csv_running_config_class

tsut.components.nodes.data_sources.tabular_csv_fetcher.TabularCSVFetcherRunningConfig

In [5]:
data_csv_config = csv_config_class()
target_csv_config = csv_config_class()

In [6]:
data_csv_config.running_config.csv_path = "../data/fake_batch.csv"
data_csv_config.running_config.context_path = "../data/fake_batch_context.json"

target_csv_config.running_config.csv_path = "../data/fake_target_df.csv"
target_csv_config.running_config.context_path = "../data/fake_target_context.json"

## 3. Configuring Transform Nodes

Transforms have both **hyperparameters** (e.g. `threshold`) and **running configs**. We can also override port modes -- for instance, the target missing-rate filter should only be active during training and evaluation (not inference, since we don't have ground-truth labels at inference time).

In [7]:
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig

In [8]:
missing_filter_config_class = NODE_REGISTRY.get_node_config_class("MissingRateFilter")
data_missing_filter_config = missing_filter_config_class()
target_missing_filter_config = missing_filter_config_class()

target_missing_filter_config.in_ports["input"].mode=["training", "evaluation"]
target_missing_filter_config.hyperparameters.threshold = 0.0

## 4. Assembling the Pipeline

A pipeline is a **directed acyclic graph (DAG)** defined by:
- **`nodes`** -- a dict mapping node names to `(node_type_name, config)` tuples. For nodes where you don't need custom configuration, you can pass a default config from the registry.
- **`edges`** -- a list of `Edge` objects that connect output ports of one node to input ports of another via `ports_map`.

The pipeline below implements a full ML workflow:
1. Load features and targets from CSV
2. Filter missing values, split by data category (numerical vs categorical)
3. Remove outliers (IQR), filter low-variance features, scale, one-hot encode
4. Concatenate processed features, feed into a `RandomForestRegressor`
5. Evaluate with R2 and MSE metrics

In [9]:
nodes ={
        "data_source": ("TabularCSVFetcher", data_csv_config),
        "target_source": ("TabularCSVFetcher", target_csv_config),
        "data_missing_filter": ("MissingRateFilter", data_missing_filter_config),
        "category_splitter": ("DataCategoryFilter", NODE_REGISTRY.get_node_config_class("DataCategoryFilter")()),
        "one_hot_encoder": ("OneHotEncoding", NODE_REGISTRY.get_node_config_class("OneHotEncoding")()),
        "target_missing_filter": ("MissingRateFilter", target_missing_filter_config),
        "iqr_filter": ("IQROutlierFilter", NODE_REGISTRY.get_node_config_class("IQROutlierFilter")()),
        "variance_threshold_filter": ("VarianceFilter", NODE_REGISTRY.get_node_config_class("VarianceFilter")()),
        "standard_scaler": ("StandardScaler", NODE_REGISTRY.get_node_config_class("StandardScaler")()),
        "feature_concatenate": ("FeatureConcatenate", NODE_REGISTRY.get_node_config_class("FeatureConcatenate")()),
        "random_forest_regressor": ("RandomForestRegressor", NODE_REGISTRY.get_node_config_class("RandomForestRegressor")()),
        "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
        "mse": ("MSE", NODE_REGISTRY.get_node_config_class("MSE")()),
        "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

In [10]:
edges = [
    Edge(source="data_source", target="data_missing_filter", ports_map=[("output", "input")]),
    Edge(source="data_missing_filter", target="category_splitter", ports_map=[("output", "input")]),
    Edge(source="one_hot_encoder", target="feature_concatenate", ports_map=[("output", "input_2")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("numerical", "input")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("categorical", "sliced")]),
    Edge(source="target_source", target="iqr_filter", ports_map=[("output", "target")]),
    Edge(source="iqr_filter", target="variance_threshold_filter", ports_map=[("output", "input")]),
    Edge(source="iqr_filter", target="one_hot_encoder", ports_map=[("sliced", "input")]),
    Edge(source="variance_threshold_filter", target="standard_scaler", ports_map=[("output", "input")]),
    Edge(source="standard_scaler", target="feature_concatenate", ports_map=[("output", "input_1")]),
    Edge(source="feature_concatenate", target="random_forest_regressor", ports_map=[("output", "X")]),
    Edge(source="iqr_filter", target="target_missing_filter", ports_map=[("target", "input")]),
    Edge(source="target_missing_filter", target="random_forest_regressor", ports_map=[("output", "y")]),
    Edge(source="random_forest_regressor", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="r2", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="mse", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="mse", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="sink", ports_map=[("pred", "dump")]),
]

In [11]:
pipe_conf = PipelineConfig(
    nodes=nodes,
    edges=edges,
)

pipe = Pipeline(config=pipe_conf)

08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_source  target=iqr_filter  source_port=output  target_port=target
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=iqr_filter  target=one_hot_encoder  source_port=sliced  target_port=input
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=random_forest_regressor  source_port=output  target_port=y
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=r2  source_port=output  target_port=target
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=mse  source_port=output  target_port=target
08:43:07 WARNING  [tsut.pipeline] Data structure mismatch (compatible but not identical)  source=random_forest_regressor  target=sink  source_port=

### Visualising the graph

`pipe.render()` produces an interactive Plotly visualisation of the DAG so you can verify the topology before running anything.

In [12]:
pipe.render()

/Users/adrien.bolling/repositories/TimeSeriesUnifiedToolbox/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value='training', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value='evaluation', input_type=str])
  return self.__pydantic_serializer__.to_python(


## 5. Compiling and Running the Pipeline

Before execution, the pipeline must be **compiled**. Compilation validates the graph structure, resolves port compatibility, and prunes unconnected nodes.

A `SmartRunner` then drives the three execution phases:
- **`train()`** -- executes the full graph in training mode (fit + transform on every node)
- **`infer()`** -- re-executes in inference mode (transform only, no targets needed)
- **`evaluate()`** -- re-executes in evaluation mode and collects outputs from all metric nodes

In [13]:
from tsut.core.pipeline.runners.smart_runner import SmartRunner

In [14]:

pipe.compile()
runnable = SmartRunner(pipe)

08:43:07 INFO     [tsut.pipeline] Phase compilation start  pipeline_name=My Pipeline  pipeline_phase=compilation  phase_status=start
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_source  target=iqr_filter  source_port=output  target_port=target
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=iqr_filter  target=one_hot_encoder  source_port=sliced  target_port=input
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=random_forest_regressor  source_port=output  target_port=y
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=r2  source_port=output  target_port=target
08:43:07 WARNING  [tsut.pipeline] Data category superset (source broader than target)  source=target_missing_filter  target=mse  source_port=output  target_port=target
08:43:07 WARNI

In [15]:
runnable.train()

08:43:07 INFO     [tsut.runner.smart] Phase training start  pipeline_name=My Pipeline  pipeline_version=0.1.0  pipeline_phase=training  phase_status=start


Training:   0%|          | 0/12 nodes [00:00<?, ?nodes/s]

08:43:07 DEBUG    [tsut.runner.smart] Node called: sink  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=sink  pipeline_phase=training
08:43:07 DEBUG    [tsut.runner.smart] Node called: random_forest_regressor  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=random_forest_regressor  pipeline_phase=training
08:43:07 DEBUG    [tsut.runner.smart] Node called: feature_concatenate  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=feature_concatenate  pipeline_phase=training
08:43:07 DEBUG    [tsut.runner.smart] Node called: one_hot_encoder  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=one_hot_encoder  pipeline_phase=training
08:43:07 DEBUG    [tsut.runner.smart] Node called: iqr_filter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=iqr_filter  pipeline_phase=training
08:43:07 DEBUG    [tsut.runner.smart] Node called: category_splitter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=category_splitter  pipeline

In [16]:
runnable.infer()

08:43:08 INFO     [tsut.runner.smart] Phase inference start  pipeline_name=My Pipeline  pipeline_version=0.1.0  pipeline_phase=inference  phase_status=start


Inference:   0%|          | 0/10 nodes [00:00<?, ?nodes/s]

08:43:08 DEBUG    [tsut.runner.smart] Node called: sink  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=sink  pipeline_phase=inference
08:43:08 DEBUG    [tsut.runner.smart] Node called: random_forest_regressor  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=random_forest_regressor  pipeline_phase=inference
08:43:08 DEBUG    [tsut.runner.smart] Node called: feature_concatenate  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=feature_concatenate  pipeline_phase=inference
08:43:08 DEBUG    [tsut.runner.smart] Node called: one_hot_encoder  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=one_hot_encoder  pipeline_phase=inference
08:43:08 DEBUG    [tsut.runner.smart] Node called: iqr_filter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=iqr_filter  pipeline_phase=inference
08:43:08 DEBUG    [tsut.runner.smart] Node called: category_splitter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=category_splitter  pip

{'dump': (     output_batch_id  LOSS-VAC  FLOW-FAC     DBULK     PRESSURE  SHRINKAG  \
  0               1327  1.787367  3.886120  3.419060  1286.273444        21   
  1               1286  1.761888  3.781065  3.395401  1276.009103        21   
  2               1141  1.733859  3.924410  3.407863  1363.687739        21   
  3               1139  1.776953  4.109279  3.407986  1353.843624        21   
  4               1186  1.785728  3.927200  3.426905  1432.082811        21   
  ..               ...       ...       ...       ...          ...       ...   
  197             1219  1.782996  3.853499  3.415032  1352.149901        21   
  198             1423  1.779433  4.321590  3.394881  1399.849942        21   
  199             1399  1.753346  4.120289  3.370126  1387.923015        21   
  200             1929  1.718055  4.496036  3.452325  1438.889460        21   
  201             1954  1.722176  4.361960  3.432497  1468.474066        21   
  
       DPRES_LU     4PS (G)    DSINTER   

### Inference

During inference, ports marked with `mode=["training", "evaluation"]` (like the `y` input on the model) are automatically skipped. The runner only computes what is needed to reach the sink node.

In [17]:
eval_metrics = runnable.evaluate()

08:43:08 INFO     [tsut.runner.smart] Phase evaluation start  pipeline_name=My Pipeline  pipeline_version=0.1.0  pipeline_phase=evaluation  phase_status=start


Evaluation:   0%|          | 0/13 nodes [00:00<?, ?nodes/s]

08:43:08 DEBUG    [tsut.runner.smart] Node called: r2  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=r2  pipeline_phase=evaluation
08:43:08 DEBUG    [tsut.runner.smart] Node called: random_forest_regressor  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=random_forest_regressor  pipeline_phase=evaluation
08:43:08 DEBUG    [tsut.runner.smart] Node called: feature_concatenate  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=feature_concatenate  pipeline_phase=evaluation
08:43:08 DEBUG    [tsut.runner.smart] Node called: one_hot_encoder  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=one_hot_encoder  pipeline_phase=evaluation
08:43:08 DEBUG    [tsut.runner.smart] Node called: iqr_filter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=iqr_filter  pipeline_phase=evaluation
08:43:08 DEBUG    [tsut.runner.smart] Node called: category_splitter  pipeline_name=My Pipeline  pipeline_version=0.1.0  node_name=category_splitter  pi

In [18]:
eval_metrics

{'r2': (        r2
  0  0.91965,
  TabularDataContext(columns=['r2'], dtypes=[dtype('float64')], categories=[<class 'tsut.core.common.data.data.NumericalData'>])),
 'mse': (           mse
  0  3908.531738,
  TabularDataContext(columns=['mse'], dtypes=[dtype('float64')], categories=[<class 'tsut.core.common.data.data.NumericalData'>]))}

In [19]:
eval_metrics["r2"]

(        r2
 0  0.91965,
 TabularDataContext(columns=['r2'], dtypes=[dtype('float64')], categories=[<class 'tsut.core.common.data.data.NumericalData'>]))

### Evaluation

`evaluate()` runs the graph in evaluation mode and returns a dict mapping each metric node name to its computed `TabularData` output. Here we get both R2 and MSE scores.